# Corpus smoke test: 10 questions qwen3:14b got wrong without the corpus

A ~20-minute check to run **before** a long eval. It answers three questions, in order, and stops at the
first "no":

1. **Is the corpus installed?** Finds the vector database in your attached dataset (the folders holding
   `chroma.sqlite3`), installs it and checks each category's chunk count against `_build_info.json`.
   Stops with an error if nothing is found -- no silent "no context" run.
2. **Does retrieval find the right law?** For each question, searches the corpus directly (no model yet)
   and checks whether the law the gold answer cites is among the hits.
3. **Does qwen3:14b answer better with it?** Answers the 10 questions with the full pipeline (plan +
   retrieval + answer, thinking on) and shows each answer next to the gold answer, with what was retrieved.

The 10 questions all scored 0 for qwen3:14b in the 28 Sept run, where the corpus was missing:

| id | category | the gold answer, in short | law |
|---|---|---|---|
| IL-001 | rule recall | offer and acceptance | Contracts (General Part) s.1 |
| IL-061 | rule recall | enforcement or rescission, plus damages | Contracts (Remedies) s.2 |
| IL-113 | rule recall | completed by registration | Land Law s.7 |
| IL-247 | rule recall | one month's notice | Advance Notice Law s.3 |
| IL-294 | rule recall | criminal responsibility from 12 | Penal Law s.34ו |
| IL-297 | rule recall | reckless killing: 12 years | Penal Law s.301ג |
| IL-212 | yes/no | no: fault doesn't reduce road-accident compensation | Road Accident Victims Law s.2 |
| IL-262 | yes/no | no: 125% for 2 hours, then 150% | Hours of Work and Rest s.16 |
| IL-040 | interpretation | what עושק means | Contracts (General Part) s.18 |
| IL-110 | citation | where revoking a gift promise is regulated | Gift Law s.5 |

## Before running (panel on the right)
1. *Add Input*: the dataset that holds **only** `legal_corpus_vectordb` (zip or unpacked -- either works).
2. *Settings -> Accelerator*: **GPU T4 x2**. *Settings -> Internet*: **On**.
3. *Add-ons -> Secrets*: **GITHUB_TOKEN** (reads `zananiri/AI-IZ`).
4. *Run All* (interactive is fine -- about 20 minutes).

In [ ]:
BRANCH = "main"
MODEL = "qwen3:14b"
THINKING = True
QUESTION_IDS = ["IL-001", "IL-061", "IL-113", "IL-247", "IL-294", "IL-297", "IL-212", "IL-262", "IL-040", "IL-110"]
TOP_K = 12                 # chunks per question, as in the full eval
CONTEXT_LENGTH = 16384
ANSWER_MAX_TOKENS = 6144

## 1. Code and packages

In [ ]:
import os, pathlib, shutil, subprocess, json, sys, zipfile
from kaggle_secrets import UserSecretsClient

def sh(command, check=True):
    """Runs a shell command with its output streamed here; raises on failure."""
    proc = subprocess.Popen(["bash", "-o", "pipefail", "-c", command], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() and check:
        raise RuntimeError(f"exit code {proc.returncode}: {command}")

REPO = "/kaggle/working/AI-IZ"
if not pathlib.Path(REPO, ".git").exists():
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    f"https://{token}@github.com/zananiri/AI-IZ.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/zananiri/AI-IZ.git"], check=True)
os.chdir(REPO)
sh("git log --oneline -1")

## 2. Find and install the corpus -- stops here if it isn't there

Looks everywhere under `/kaggle/input` for category folders that hold a Chroma database (`chroma.sqlite3`),
unpacked or inside a zip, so the dataset's folder layout doesn't matter. Folders of `.jsonl` records
(the build's `legal_corpus_jsonl`) are ignored: they are the texts, not the searchable database.

In [ ]:
INPUT = pathlib.Path("/kaggle/input")
VECTORDB = pathlib.Path("data/legal_corpus_vectordb")
CATEGORIES_WANTED = ("laws", "procedural_rules")

# Unpacked: a folder whose laws/ or procedural_rules/ holds chroma.sqlite3.
roots = sorted({p.parent.parent for p in INPUT.rglob("chroma.sqlite3") if p.parent.name in CATEGORIES_WANTED})
zips = sorted(INPUT.rglob("*.zip"))
print("database folders found:", [str(r) for r in roots] or "none", "| zips found:", [z.name for z in zips] or "none")

shutil.rmtree(VECTORDB, ignore_errors=True)
VECTORDB.mkdir(parents=True)
if roots:
    src = roots[0]
    for item in src.iterdir():
        dest = VECTORDB / item.name
        (shutil.copytree if item.is_dir() else shutil.copy2)(item, dest)
    print("copied", src, "->", VECTORDB)
else:
    # A zip: find the one holding category databases, and unpack it without any extra folder level.
    for z in zips:
        with zipfile.ZipFile(z) as archive:
            names = archive.namelist()
            dbs = [n for n in names if n.endswith("chroma.sqlite3") and n.split("/")[-2] in CATEGORIES_WANTED]
            if not dbs:
                continue
            parts = dbs[0].split("/")                       # [..., <category>, "chroma.sqlite3"]
            prefix = "/".join(parts[:-2]) + "/" if len(parts) > 2 else ""
            for member in archive.infolist():
                if member.filename.startswith(prefix) and not member.filename.endswith("/") and ".." not in member.filename:
                    dest = VECTORDB / member.filename[len(prefix):]
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(member) as fsrc, open(dest, "wb") as fdst:
                        shutil.copyfileobj(fsrc, fdst)
            print("unpacked", z.name, "->", VECTORDB)
            break

CATEGORIES = sorted(c for c in CATEGORIES_WANTED if (VECTORDB / c / "chroma.sqlite3").exists())
assert CATEGORIES, ("NO CORPUS DATABASE FOUND under /kaggle/input: attach the dataset with legal_corpus_vectordb "
                    "(its laws/ and procedural_rules/ folders must contain chroma.sqlite3)")
BUILD_INFO = json.loads((VECTORDB / "_build_info.json").read_text()) if (VECTORDB / "_build_info.json").exists() else {}
print("categories:", CATEGORIES, "| built", BUILD_INFO.get("built_at"), "with chromadb", BUILD_INFO.get("chromadb_version"))

pin = f' "chromadb=={BUILD_INFO["chromadb_version"]}"' if BUILD_INFO.get("chromadb_version") else ""
sh(f'pip install -q -e ".[legal,legal-data]"{pin}')

In [ ]:
# Open every collection and compare its chunk count with the build's record.
import chromadb
prefix = "israeli_law_corpus"
problems = 0
for c in CATEGORIES:
    col = chromadb.PersistentClient(path=str(VECTORDB / c)).get_collection(f"{prefix}_{c}")
    n = col.count()
    expected = BUILD_INFO.get("categories", {}).get(c, {}).get("chunks")
    ok = n > 0 and (expected is None or n == expected)
    problems += not ok
    print(f"{c:18s} {n:>9,} chunks" + (f"  (build says {expected:,})" if expected else "") + ("  OK" if ok else "  <-- PROBLEM"))
assert not problems, "a collection is empty or its count doesn't match the build: rebuild or re-upload the database"

## 3. Retrieval alone (no model yet): is the right law among the hits?

A plain search on each question's text (the eval also plans issues first, which usually helps more). A
question passes when a hit comes from the law the gold answer cites.

In [ ]:
EVAL_DIR = pathlib.Path("legal_txt/Evals")
zipfile.ZipFile(EVAL_DIR / "israeli_legal_eval.zip").extractall(EVAL_DIR)
EVAL = EVAL_DIR / "israeli_legal_eval"
load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
QS = {q["id"]: q for q in load(EVAL / "questions.jsonl")}
GOLD = {g["id"]: g for g in load(EVAL / "gold.jsonl")}
QUESTIONS = pathlib.Path("/kaggle/working/smoke_questions.jsonl")
with open(QUESTIONS, "w", encoding="utf-8") as f:
    for i in QUESTION_IDS:
        f.write(json.dumps(QS[i], ensure_ascii=False) + "\n")

probe = r"""
import json, sys
sys.path.insert(0, "scripts/legal_data")
from eval_run import retrieve, _question_text
from docslides.legal.corpus_retrieval import same_law
cats, top_k = sys.argv[1].split(","), int(sys.argv[2])
out = {}
for line in open(sys.argv[3], encoding="utf-8"):
    q = json.loads(line)
    hits = retrieve(_question_text(q), cats, top_k)
    out[q["id"]] = [{"title": h["meta"].get("title"), "section": h["meta"].get("section_number"),
                     "distance": round(h["distance"], 3)} for h in hits]
json.dump(out, open(sys.argv[4], "w", encoding="utf-8"), ensure_ascii=False)
"""
pathlib.Path("/kaggle/working/probe.py").write_text(probe)
sh(f'CUDA_VISIBLE_DEVICES= python /kaggle/working/probe.py {",".join(CATEGORIES)} {TOP_K} {QUESTIONS} /kaggle/working/probe_hits.json')
PROBE = json.load(open("/kaggle/working/probe_hits.json", encoding="utf-8"))

sys.path.insert(0, "src")
from docslides.legal.corpus_retrieval import same_law
found = 0
for i in QUESTION_IDS:
    laws = [c["law"] for c in GOLD[i]["citations"]]
    ok = any(same_law(law, h["title"] or "") for law in laws for h in PROBE[i])
    found += ok
    print(f"{'PASS' if ok else 'MISS'}  {i}  gold law: {laws[0]}")
    for h in PROBE[i][:4]:
        print(f"        {h['distance']:.3f}  {h['title']}  s.{h['section']}")
print(f"\nthe gold law is among the plain-search hits for {found} of {len(QUESTION_IDS)} questions")
assert found > 0, "retrieval found none of the gold laws: the corpus is installed but not searchable -- stop here"

## 4. Ollama and qwen3:14b

In [ ]:
sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null")
if not shutil.which("ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh > /dev/null")
import time, urllib.request
subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT,
                 env={**os.environ, "OLLAMA_MAX_LOADED_MODELS": "1"})
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(2)
sh(f"ollama pull {MODEL}")

## 5. Answer the 10 questions with the full pipeline

The same script and settings as the big eval: the model lists the issues and laws, the corpus is searched
on each, and the answer is written from the retrieved provisions.

In [ ]:
import yaml
cfg = yaml.safe_load(open("config/config.yaml", encoding="utf-8"))
cfg["legal"]["orchestrator"].update(backend="ollama", base_url="http://localhost:11434", model=MODEL,
                                    max_model_len=CONTEXT_LENGTH, request_timeout_s=1800)
pathlib.Path("config/smoke.yaml").write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
# Retrieval (bge-m3 + the reranker) on the CPU, exactly as the full eval notebooks run it: Ollama keeps the
# GPU for the model. On the GPU, the reranker ran out of memory next to qwen3:14b and was silently skipped.
env = {"DOCSLIDES_CONFIG": "config/smoke.yaml", "DOCSLIDES_LLM_TRACE_DIR": "/kaggle/working/llm_trace",
       "CUDA_VISIBLE_DEVICES": ""}
os.environ.update(env)
ANSWERS = pathlib.Path("/kaggle/working/smoke_answers.jsonl")
ANSWERS.unlink(missing_ok=True)
sh(f'python scripts/legal_data/eval_run.py answer --questions {QUESTIONS} --categories {",".join(CATEGORIES)} '
   f'--top-k {TOP_K} --max-tokens {ANSWER_MAX_TOKENS} --out {ANSWERS}{"" if THINKING else " --no-thinking"}')

## 6. Results: each answer next to the gold answer

For each question: whether the gold law was retrieved (by the full pipeline this time), the plan, the top
retrieved provisions, the answer and the gold answer. Judging the answers is for review (send the zip).

In [ ]:
ANS = {a["id"]: a for a in load(ANSWERS)}
retrieved_ok = 0
lines = ["# Corpus smoke test", "", f"model {MODEL}, categories {CATEGORIES}, top_k {TOP_K}", ""]
for i in QUESTION_IDS:
    a, g = ANS[i], GOLD[i]
    laws = [c["law"] for c in g["citations"]]
    ok = any(same_law(law, h.get("title") or "") for law in laws for h in a["retrieved"])
    retrieved_ok += ok
    block = [f"## {i} -- gold law {'RETRIEVED' if ok else 'NOT retrieved'}",
             f"**Question:** {QS[i]['question']}",
             f"**Plan:** " + "; ".join(f"{p['law']} {p.get('sections')}" for p in a.get("plan", [])),
             "**Top retrieved:** " + "; ".join(f"{h.get('title')} s.{h.get('section')}" for h in a["retrieved"][:5]),
             f"**Answer:** {a['answer'] or '(empty)'}" + (f"  \n_error: {a['error']}_" if a.get("error") else ""),
             f"**Gold:** {g['gold_answer']}  (cites {', '.join(laws)})", ""]
    lines += block
    print("\n".join(block))
summary = f"gold law retrieved by the pipeline for {retrieved_ok} of {len(QUESTION_IDS)} questions"
print("=" * 80 + "\n" + summary)
lines.insert(3, summary)
pathlib.Path("/kaggle/working/smoke_report.md").write_text("\n".join(lines), encoding="utf-8")

## 7. Package for review

In [ ]:
sh("cd /kaggle/working && zip -qr corpus_smoke_results.zip smoke_report.md smoke_answers.jsonl probe_hits.json "
   "llm_trace ollama.log AI-IZ/config/smoke.yaml && ls -lh corpus_smoke_results.zip")
print("Download corpus_smoke_results.zip from the Output panel and send it for review.")